# 🤖 Laboratório de RPA — Robô de Contas a Pagar
**MBA em Desenvolvimento Fullstack · Disciplina: RPA — Automação de Processos Robóticos**
Prof. MSc. Kaio Mesquita · Aula prática (turno da tarde)

---

## O cenário

A **TransVia Logística** (empresa fictícia de Fortaleza) recebe por e-mail cerca de **800 notas fiscais de serviço por mês**, em PDF.
Hoje, um analista de Contas a Pagar faz, para cada nota:

1. abre o PDF;
2. confere se o CNPJ do fornecedor é válido e se a nota já não foi lançada;
3. copia os campos e digita no **ERP legado** (um sistema web antigo, **sem API**);
4. anota o protocolo numa planilha de controle;
5. no fim do dia, manda um resumo para a gerência.

São ~4 minutos por nota, ou **~53 horas por mês** de trabalho repetitivo — e com erros de digitação.

## O que vamos construir

Um **robô de software** que faz exatamente o que o analista faz, pela interface gráfica, seguindo o padrão de mercado
**Dispatcher → Performer** com tratamento de exceções no estilo do **REFramework** (UiPath):

```
 ┌──────────┐   ┌─────────────┐   ┌──────────┐   ┌──────────────┐   ┌──────────┐
 │  INIT    │──▶│ GET ITEM    │──▶│ PROCESS  │──▶│ SET STATUS   │──▶│   END    │
 │ config,  │   │ próximo PDF │   │ extrair, │   │ sucesso /    │   │ relatório│
 │ login    │   │ da fila     │   │ validar, │   │ exc. negócio/│   │ + e-mail │
 └──────────┘   └─────────────┘   │ lançar   │   │ exc. sistema │   └──────────┘
                      ▲           └──────────┘   └──────┬───────┘
                      └─────────────────────────────────┘  (loop até a fila esvaziar)
```

| Parte | O que faz | Conceito de RPA |
|---|---|---|
| 0 | Instalação e configuração | Ambiente, credenciais, *assets* |
| 1 | Gera 12 NFs em PDF (com "pegadinhas") | Dados de entrada realistas |
| 2 | Sobe um **ERP legado** local | Sistema-alvo sem API |
| 3 | Extrai dados do PDF | *Document understanding* (regras) |
| 4 | Regras de negócio | Exceção de **negócio** × de **sistema** |
| 5 | Robô no navegador (Playwright) | Automação de UI, seletores, esperas |
| 6 | Orquestração com retentativas | REFramework, idempotência, *human-in-the-loop* |
| 7 | Relatório Excel + métricas de ROI | Entregável para o negócio |
| 8 | Auditoria / reconciliação | Governança e rastreabilidade |
| 9 | **Atividade avaliativa** (5 desafios com células prontas) | Sua vez |

> 💡 **Por que Python e não UiPath/Power Automate?** Os conceitos são os mesmos (seletores, filas, exceções, orquestração).
> Fazendo "na mão", vocês enxergam o que as plataformas escondem — e, como devs fullstack, saberão quando uma plataforma vale a licença.

## Parte 0 — Instalação e configuração

A célula abaixo **confere e instala automaticamente** tudo o que falta (pode rodar quantas vezes quiser).
Na primeira vez ela baixa o navegador Chromium do Playwright, o que leva 1–3 minutos.

| Biblioteca (pip) | Papel no robô |
|---|---|
| `playwright` | "mãos e olhos" do robô no navegador |
| `pypdf` | ler o texto dos PDFs |
| `fpdf2` | gerar as NFs de teste |
| `pandas` + `openpyxl` | tabela de resultados e relatório Excel |

O resto (`http.server`, `asyncio`, `logging`, `re`...) já vem com o Python. Requer **Python 3.10 ou superior**.

In [1]:
# ============================ Instalação automática ============================
import sys, subprocess, importlib

assert sys.version_info >= (3, 10), f"Use Python 3.10 ou superior (você está no {sys.version.split()[0]})"

# nome usado no import  →  nome do pacote no pip
NECESSARIOS = {"playwright": "playwright", "pypdf": "pypdf", "fpdf": "fpdf2",
               "pandas": "pandas", "openpyxl": "openpyxl"}

faltando = []
for modulo, pacote in NECESSARIOS.items():
    try:
        importlib.import_module(modulo)
    except ImportError:
        faltando.append(pacote)

if faltando:
    print("📦 Instalando:", ", ".join(faltando))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltando], check=True)
    importlib.invalidate_caches()

# Baixa o Chromium do Playwright (se já existir, termina em segundos)
print("🌐 Conferindo o navegador do Playwright...")
r = subprocess.run([sys.executable, "-m", "playwright", "install", "chromium"],
                   capture_output=True, text=True)
if r.returncode != 0:
    print(r.stdout[-1500:], r.stderr[-1500:])
    raise RuntimeError("Falha ao instalar o Chromium. Rode no terminal: python -m playwright install chromium")

for modulo in NECESSARIOS:
    m = importlib.import_module(modulo)
    print(f"  ✔ {modulo:<10} {getattr(m, '__version__', '')}")
print("✅ Ambiente pronto." + (" Se algo foi instalado agora e der erro de import, reinicie o kernel." if faltando else ""))

🌐 Conferindo o navegador do Playwright...
  ✔ playwright 
  ✔ pypdf      6.20.0
  ✔ fpdf       2.8.9
  ✔ pandas     1.5.3
  ✔ openpyxl   3.0.10
✅ Ambiente pronto.


In [ ]:
# ===================== CONFIGURAÇÃO DO ROBÔ ("Config.xlsx" do REFramework) =====================
from pathlib import Path
import os, sys, re, json, time, random, asyncio, logging, threading, datetime as dt
from dataclasses import dataclass, asdict, field

# Pastas de trabalho — num robô real seriam pastas de rede ou um bucket
BASE           = Path("lab_rpa")
PASTA_ENTRADA  = BASE / "entrada"      # onde chegam os PDFs (caixa de e-mail "baixada")
PASTA_SAIDA    = BASE / "saida"        # relatórios gerados
PASTA_EVID     = BASE / "evidencias"   # prints de tela para auditoria
for p in (PASTA_ENTRADA, PASTA_SAIDA, PASTA_EVID):
    p.mkdir(parents=True, exist_ok=True)

# Sistema-alvo
ERP_HOST, ERP_PORTA = "127.0.0.1", 8765
ERP_URL = f"http://{ERP_HOST}:{ERP_PORTA}"

# ⚠️ Credenciais: NUNCA deixe senha no código em produção.
# Use um cofre (UiPath Orchestrator Assets, CyberArk, Azure Key Vault) ou variáveis de ambiente.
ERP_USUARIO = os.getenv("ERP_USUARIO", "robo.ap")
ERP_SENHA   = os.getenv("ERP_SENHA",   "Rpa@2026")

# Regras de negócio parametrizáveis (mudam sem mexer no código)
LIMITE_APROVACAO   = 50_000.00   # acima disso, a nota vai para aprovação humana
MAX_TENTATIVAS     = 3           # retentativas para exceções de SISTEMA
HEADLESS           = True        # False = você VÊ o robô trabalhando (ótimo para a aula!)
LENTIDAO_MS        = 0           # ex.: 400 para "câmera lenta" quando HEADLESS = False

# Parâmetros do "ERP legado" de laboratório
FALHA_ALEATORIA_ERP = 0.20       # 20% das requisições devolvem "sistema indisponível"
LAYOUT_ERP          = "v1"       # "v2" muda os IDs da tela (usado no Desafio 4)

random.seed(2026)                # reprodutibilidade das NFs geradas

# Log estruturado: tela + arquivo (o log é a "caixa-preta" do robô)
logging.basicConfig(
    level=logging.INFO, force=True,
    format="%(asctime)s | %(levelname)-7s | %(message)s", datefmt="%H:%M:%S",
    handlers=[logging.StreamHandler(), logging.FileHandler(BASE / "robo.log", encoding="utf-8")],
)
log = logging.getLogger("robo-ap")
log.info("Configuração carregada. ERP em %s | limite de aprovação R$ %.2f", ERP_URL, LIMITE_APROVACAO)

## Parte 1 — Gerando as notas fiscais de entrada

Em vez de usar dados reais (LGPD!), vamos **gerar 12 NFs fictícias em PDF**. Quatro delas têm problemas de propósito —
é assim que o mundo real chega para o robô:

| NF | Problema plantado | O que o robô deve fazer |
|---|---|---|
| 4 | CNPJ com dígito verificador errado | Exceção de **negócio** → não lança |
| 7 | Mesma nota da NF 2 (duplicada) | Exceção de **negócio** → não lança |
| 9 | Valor acima do limite | **Fila de aprovação humana** |
| 11 | PDF "mal escaneado": sem o campo valor | Exceção de **negócio** (extração) |

Primeiro, o algoritmo do **CNPJ** — útil para gerar dados de teste e, depois, para validar.

In [ ]:
# ============================ CNPJ: geração e validação ============================
# Regra da Receita: 12 caracteres-base + 2 dígitos verificadores (DV), módulo 11.
PESOS_DV1 = [5, 4, 3, 2, 9, 8, 7, 6, 5, 4, 3, 2]
PESOS_DV2 = [6] + PESOS_DV1

def _calcular_dv(base: str, pesos: list[int]) -> str:
    # Cada caractere vale (código ASCII - 48): '0'..'9' → 0..9.
    # (Isso já deixa o terreno pronto para o CNPJ alfanumérico — veja o Desafio 1.)
    soma = sum((ord(c) - 48) * p for c, p in zip(base, pesos))
    resto = soma % 11
    return "0" if resto < 2 else str(11 - resto)

def gerar_cnpj() -> str:
    base = f"{random.randint(10_000_000, 99_999_999)}0001"   # raiz (8) + ordem 0001 = matriz
    dv1 = _calcular_dv(base, PESOS_DV1)
    dv2 = _calcular_dv(base + dv1, PESOS_DV2)
    return formatar_cnpj(base + dv1 + dv2)

def formatar_cnpj(c: str) -> str:
    return f"{c[:2]}.{c[2:5]}.{c[5:8]}/{c[8:12]}-{c[12:]}"

def limpar_cnpj(cnpj: str) -> str:
    return re.sub(r"[^0-9]", "", cnpj)       # ⚠️ só dígitos — premissa que vai quebrar no Desafio 1

def cnpj_valido(cnpj: str) -> bool:
    c = limpar_cnpj(cnpj)
    if len(c) != 14 or c == c[0] * 14:       # "00000000000000" passa no DV, mas é inválido
        return False
    dv1 = _calcular_dv(c[:12], PESOS_DV1)
    dv2 = _calcular_dv(c[:12] + dv1, PESOS_DV2)
    return c[12:] == dv1 + dv2

# Teste rápido — sempre teste suas funções antes de colocá-las num robô!
assert cnpj_valido("23.775.684/0001-07") and not cnpj_valido("23.775.684/0001-08")
print("✔ validação de CNPJ OK")

In [ ]:
# ============================ Massa de dados: 12 NFs ============================
FORNECEDORES = [
    "Sinaliza Ceará Engenharia LTDA",
    "Asfalto Norte Pavimentação S.A.",
    "LogMov Transportes e Fretes LTDA",
    "Ponte Forte Consultoria em Obras ME",
    "TecVia Sistemas de Monitoramento LTDA",
    "Iracema Manutenção Predial LTDA",
]
CNPJ_FORNECEDOR = {f: gerar_cnpj() for f in FORNECEDORES}

SERVICOS = {
    "OPERACOES":  ["Manutenção preventiva de frota", "Frete de transferência Fortaleza-Recife", "Lavagem técnica de caminhões"],
    "TI":         ["Licença mensal de rastreamento GPS", "Suporte técnico ao sistema de roteirização"],
    "ENGENHARIA": ["Sinalização horizontal do pátio", "Laudo estrutural de galpão", "Recapeamento de acesso interno"],
    "ADMINISTRATIVO": ["Manutenção de ar-condicionado", "Serviço de limpeza e conservação"],
}

def nova_nf(numero: int) -> dict:
    fornecedor = random.choice(FORNECEDORES)
    cc = random.choice(list(SERVICOS))
    return {
        "numero": f"{numero:06d}",
        "data_emissao": (dt.date(2026, 10, 1) + dt.timedelta(days=random.randint(0, 9))).strftime("%d/%m/%Y"),
        "prestador": fornecedor,
        "cnpj": CNPJ_FORNECEDOR[fornecedor],
        "descricao": random.choice(SERVICOS[cc]),
        "valor": round(random.uniform(850, 38_000), 2),
        "centro_custo": cc,
    }

notas_geradas = [nova_nf(4100 + i) for i in range(1, 13)]   # NF 1..12

# --- Pegadinhas (índice = NF - 1) ---
cnpj_ruim = notas_geradas[3]["cnpj"]
notas_geradas[3]["cnpj"] = cnpj_ruim[:-1] + ("0" if cnpj_ruim[-1] != "0" else "1")   # NF 4: DV errado
notas_geradas[6] = dict(notas_geradas[1])                                             # NF 7: duplicada da NF 2
notas_geradas[8]["valor"] = 78_450.90                                                 # NF 9: acima do limite
notas_geradas[10]["valor"] = None                                                     # NF 11: campo ilegível

def brl(v: float) -> str:
    # 12345.6 → "12.345,60" (formato brasileiro)
    return f"{v:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")

import pandas as pd
pd.DataFrame(notas_geradas).assign(valor=lambda d: d["valor"].map(lambda v: brl(v) if v else "— ilegível —"))

In [ ]:
# ============================ Gerar os PDFs ============================
from fpdf import FPDF
# Compatível com fpdf2 (recomendado) e com o antigo "fpdf" 1.7: usamos cell() + ln()
# em vez dos parâmetros new_x/new_y, que só existem no fpdf2.

def _linha(pdf, altura, texto, **kw):
    pdf.cell(0, altura, texto, **kw)
    pdf.ln(altura)                                     # pula para a próxima linha

def gerar_pdf_nf(nf: dict, caminho: Path) -> None:
    pdf = FPDF()
    pdf.add_page()
    pdf.set_font("Helvetica", "B", 15)
    _linha(pdf, 10, "NOTA FISCAL DE SERVIÇOS ELETRÔNICA - NFS-e", align="C")
    pdf.set_font("Helvetica", "", 9)
    _linha(pdf, 6, "Prefeitura Municipal de Fortaleza - Documento fictício para fins didáticos", align="C")
    pdf.ln(6)

    linhas = [
        ("Número da Nota", nf["numero"]),
        ("Data de Emissão", nf["data_emissao"]),
        ("Prestador", nf["prestador"]),
        ("CNPJ do Prestador", nf["cnpj"]),
        ("Tomador", "TransVia Logística LTDA"),
        ("Discriminação do Serviço", nf["descricao"]),
        ("Centro de Custo", nf["centro_custo"]),
    ]
    if nf["valor"] is not None:                        # NF 11 sai sem valor ("scan ruim")
        linhas.append(("Valor Total da Nota", f"R$ {brl(nf['valor'])}"))

    for rotulo, valor in linhas:
        pdf.set_font("Helvetica", "B", 11); pdf.cell(60, 9, f"{rotulo}:")
        pdf.set_font("Helvetica", "", 11);  _linha(pdf, 9, str(valor))
    pdf.output(str(caminho))

# Limpa a "caixa de entrada" e gera os arquivos
for antigo in PASTA_ENTRADA.glob("*.pdf"):
    antigo.unlink()
for i, nf in enumerate(notas_geradas, start=1):
    gerar_pdf_nf(nf, PASTA_ENTRADA / f"NF_{i:02d}_{nf['numero']}.pdf")

print(f"📄 {len(list(PASTA_ENTRADA.glob('*.pdf')))} PDFs em {PASTA_ENTRADA.resolve()}")

## Parte 2 — O "ERP legado" (sistema-alvo)

RPA brilha onde **não existe API**: sistemas antigos, portais de governo, telas de fornecedores.
A célula abaixo sobe, dentro do próprio notebook, um mini-ERP web com:

- tela de **login** (com sessão por cookie);
- formulário de **lançamento de NF** com IDs "estilo anos 2000" (`txtNumNF`, `cmbCentroCusto`...);
- campos que exigem **formato brasileiro** (data `dd/mm/aaaa`, valor `12345,67`);
- **lentidão** variável e **falhas aleatórias** (HTTP 503) — o pesadelo de qualquer robô;
- listagem `/notas` com tudo que foi lançado.

👉 Depois de rodar, **abra `http://127.0.0.1:8765` no seu navegador** e faça um lançamento manual (usuário `robo.ap`, senha `Rpa@2026`).
Cronometre. Esse é o "jeito humano".

In [ ]:
# ============================ Mini-ERP legado (NÃO é parte do robô) ============================
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from urllib.parse import parse_qs, urlparse
import secrets, html

ERP_DB = {"notas": [], "sessoes": set(), "seq": 0}
CENTROS = ["OPERACOES", "TI", "ENGENHARIA", "ADMINISTRATIVO"]

# IDs dos campos por versão de layout — o v2 simula a "atualização do sistema" que quebra robôs
IDS = {
    "v1": dict(num="txtNumNF", dt="txtDtEmissao", cnpj="txtCNPJ", forn="txtFornecedor",
               desc="txtDescricao", valor="txtValor", cc="cmbCentroCusto", btn="btnGravar", btn_txt="Gravar"),
    "v2": dict(num="nfNumero", dt="nfData", cnpj="inputCnpjFornecedor", forn="nfRazaoSocial",
               desc="nfServico", valor="nfValorTotal", cc="selectCC", btn="btnSalvarLancamento", btn_txt="Salvar lançamento"),
}

CSS = ("<style>body{font-family:Tahoma,Verdana,sans-serif;background:#d4d0c8;margin:0}"
       ".top{background:#0a246a;color:#fff;padding:6px 12px;font-weight:bold}"
       ".box{background:#ece9d8;border:2px outset #fff;margin:24px auto;padding:16px;width:560px}"
       "label{display:inline-block;width:170px;font-size:13px}input,select{margin:4px 0;width:340px}"
       ".err{color:#b00;font-weight:bold}.ok{color:#060;font-weight:bold}table{border-collapse:collapse;font-size:12px}"
       "td,th{border:1px solid #888;padding:3px 6px}</style>")

def pagina(titulo, corpo):
    return f"<html><head><meta charset='utf-8'><title>{titulo}</title>{CSS}</head><body>" \
           f"<div class='top'>SIGA-ERP 4.2 &middot; TransVia Logística &middot; {titulo}</div>" \
           f"<div class='box'>{corpo}</div></body></html>"

class ERPHandler(BaseHTTPRequestHandler):
    def log_message(self, *args):          # silencia o log padrão do http.server
        pass

    # ---------- utilitários ----------
    def _responder(self, corpo, status=200, headers=None):
        dados = corpo.encode("utf-8")
        self.send_response(status)
        self.send_header("Content-Type", "text/html; charset=utf-8")
        for k, v in (headers or {}).items():
            self.send_header(k, v)
        self.send_header("Content-Length", str(len(dados)))
        self.end_headers()
        self.wfile.write(dados)

    def _redirecionar(self, destino, headers=None):
        self.send_response(303)
        self.send_header("Location", destino)
        for k, v in (headers or {}).items():
            self.send_header(k, v)
        self.end_headers()

    def _logado(self):
        cookie = self.headers.get("Cookie", "")
        m = re.search(r"SESSAO=([a-f0-9]+)", cookie)
        return bool(m and m.group(1) in ERP_DB["sessoes"])

    def _form(self):
        tam = int(self.headers.get("Content-Length", 0))
        return {k: v[0].strip() for k, v in parse_qs(self.rfile.read(tam).decode("utf-8")).items()}

    # ---------- GET ----------
    def do_GET(self):
        time.sleep(random.uniform(0.05, 0.35))            # sistema lento
        rota = urlparse(self.path).path
        if rota in ("/", "/login"):
            return self._responder(pagina("Login", """
              <h3>Acesso ao sistema</h3><form method='post' action='/login'>
              <label for='usuario'>Usuário</label><input id='usuario' name='usuario'><br>
              <label for='senha'>Senha</label><input id='senha' name='senha' type='password'><br>
              <button id='btnEntrar' type='submit'>Entrar</button></form>"""))
        if not self._logado():
            return self._redirecionar("/login")
        if rota == "/home":
            return self._responder(pagina("Início", """<p class='ok' id='boasVindas'>Bem-vindo(a)!</p>
              <a id='lnkNovaNota' href='/nova-nota'>Lançar nota fiscal</a> | <a href='/notas'>Notas lançadas</a>"""))
        if rota == "/nova-nota":
            i = IDS[LAYOUT_ERP]
            opcoes = "".join(f"<option value='{c}'>{c.title()}</option>" for c in CENTROS)
            return self._responder(pagina("Lançamento de NF", f"""
              <h3>Lançamento de Nota Fiscal de Serviço</h3><form method='post' action='/nova-nota'>
              <label for='{i['num']}'>Nº da NF</label><input id='{i['num']}' name='numero'><br>
              <label for='{i['dt']}'>Emissão (dd/mm/aaaa)</label><input id='{i['dt']}' name='data_emissao'><br>
              <label for='{i['cnpj']}'>CNPJ fornecedor</label><input id='{i['cnpj']}' name='cnpj'><br>
              <label for='{i['forn']}'>Razão social</label><input id='{i['forn']}' name='fornecedor'><br>
              <label for='{i['desc']}'>Descrição</label><input id='{i['desc']}' name='descricao'><br>
              <label for='{i['valor']}'>Valor (ex.: 1234,56)</label><input id='{i['valor']}' name='valor'><br>
              <label for='{i['cc']}'>Centro de custo</label><select id='{i['cc']}' name='centro_custo'>
                 <option value=''>-- selecione --</option>{opcoes}</select><br>
              <button id='{i['btn']}' type='submit'>{i['btn_txt']}</button></form>"""))
        if rota == "/notas":
            linhas = "".join(
                f"<tr><td>{n['protocolo']}</td><td>{n['numero']}</td><td>{n['cnpj']}</td>"
                f"<td>{html.escape(n['fornecedor'])}</td><td>{n['valor']}</td><td>{n['centro_custo']}</td></tr>"
                for n in ERP_DB["notas"])
            return self._responder(pagina("Notas lançadas", f"""<table id='tblNotas'>
              <tr><th>Protocolo</th><th>NF</th><th>CNPJ</th><th>Fornecedor</th><th>Valor</th><th>CC</th></tr>{linhas}</table>"""))
        self._responder(pagina("Erro", "<p class='err'>Página não encontrada</p>"), 404)

    # ---------- POST ----------
    def do_POST(self):
        time.sleep(random.uniform(0.1, 0.6))
        rota = urlparse(self.path).path
        if rota == "/login":
            f = self._form()
            if f.get("usuario") == ERP_USUARIO and f.get("senha") == ERP_SENHA:
                token = secrets.token_hex(8)
                ERP_DB["sessoes"].add(token)
                return self._redirecionar("/home", {"Set-Cookie": f"SESSAO={token}; Path=/"})
            return self._responder(pagina("Login", "<p class='err' id='erroLogin'>Usuário ou senha inválidos</p>"
                                          "<a href='/login'>Voltar</a>"), 401)
        if not self._logado():
            return self._redirecionar("/login")
        if rota == "/nova-nota":
            if random.random() < FALHA_ALEATORIA_ERP:          # instabilidade do legado
                return self._responder(pagina("Erro", "<p class='err' id='erroSistema'>"
                                              "Serviço temporariamente indisponível (ORA-12541). Tente novamente.</p>"), 503)
            f = self._form()
            obrig = ["numero", "data_emissao", "cnpj", "fornecedor", "descricao", "valor", "centro_custo"]
            faltando = [c for c in obrig if not f.get(c)]
            erro = None
            if faltando:
                erro = f"Campos obrigatórios não preenchidos: {', '.join(faltando)}"
            elif not re.fullmatch(r"\d{2}/\d{2}/\d{4}", f["data_emissao"]):
                erro = "Data inválida. Use dd/mm/aaaa."
            elif not re.fullmatch(r"\d+(,\d{2})?", f["valor"]):
                erro = "Valor inválido. Use apenas números e vírgula (ex.: 1234,56)."
            elif any(n["numero"] == f["numero"] and n["cnpj"] == f["cnpj"] for n in ERP_DB["notas"]):
                erro = "Nota já lançada para este fornecedor."
            if erro:
                return self._responder(pagina("Erro", f"<p class='err' id='erroValidacao'>{erro}</p>"), 422)
            ERP_DB["seq"] += 1
            protocolo = f"AP-2026-{ERP_DB['seq']:05d}"
            ERP_DB["notas"].append({**f, "protocolo": protocolo, "lancado_em": dt.datetime.now().isoformat()})
            return self._responder(pagina("Sucesso", f"<p class='ok'>Nota gravada com sucesso.</p>"
                                          f"<p>Protocolo: <b id='protocolo'>{protocolo}</b></p>"
                                          "<a href='/nova-nota'>Nova nota</a>"))
        self._responder(pagina("Erro", "<p class='err'>Rota inválida</p>"), 404)

def iniciar_erp():
    global SERVIDOR_ERP
    try:                                      # se a célula for rodada de novo, derruba o anterior
        SERVIDOR_ERP.shutdown(); SERVIDOR_ERP.server_close()
    except NameError:
        pass
    ThreadingHTTPServer.allow_reuse_address = True
    SERVIDOR_ERP = ThreadingHTTPServer((ERP_HOST, ERP_PORTA), ERPHandler)
    threading.Thread(target=SERVIDOR_ERP.serve_forever, daemon=True).start()
    log.info("🖥️  ERP legado no ar em %s  (layout %s)", ERP_URL, LAYOUT_ERP)

ERP_DB.update(notas=[], sessoes=set(), seq=0)   # base limpa a cada execução
iniciar_erp()

## Parte 3 — Extração de dados do PDF

Aqui o robô "lê" o documento. Usamos **regras (regex)** porque o layout é fixo. Em documentos variados
(notas de prefeituras diferentes, boletos, contratos), entra o **IDP — Intelligent Document Processing** (OCR + modelos de IA).

Repare em duas exceções que vamos usar o tempo todo:

- **`ErroNegocio`** (*BusinessRuleException*): o dado está errado. **Não adianta tentar de novo** — vai para um humano.
- **`ErroSistema`** (*SystemException*): a tela não carregou, o sistema caiu. **Tentar de novo resolve.**

> 🗣️ **Pergunta para a turma:** um PDF que não abre porque está corrompido é exceção de negócio ou de sistema? E se o ERP recusar a senha do robô?

In [ ]:
# ============================ Exceções e modelo de dados ============================
from pypdf import PdfReader

class ErroNegocio(Exception):
    '''Dado inválido ou regra violada. Retentar NÃO resolve → humano analisa.'''

class ErroSistema(Exception):
    '''Falha técnica/transitória (timeout, 503, elemento não carregou). Retentar PODE resolver.'''

@dataclass
class NotaFiscal:
    arquivo: str
    numero: str
    data_emissao: str
    prestador: str
    cnpj: str
    descricao: str
    centro_custo: str
    valor: float

# Um padrão por campo. (?m) = multilinha; [ \t]* evita "pular" para a linha de baixo
PADROES = {
    "numero":       r"(?m)^N[úu]mero da Nota:[ \t]*(\d+)",
    "data_emissao": r"(?m)^Data de Emiss[ãa]o:[ \t]*(\d{2}/\d{2}/\d{4})",
    "prestador":    r"(?m)^Prestador:[ \t]*(.+)$",
    "cnpj":         r"(?m)^CNPJ do Prestador:[ \t]*([\d./-]{14,18})",
    "descricao":    r"(?m)^Discrimina[çc][ãa]o do Servi[çc]o:[ \t]*(.+)$",
    "centro_custo": r"(?m)^Centro de Custo:[ \t]*([A-Z]+)",
    "valor":        r"(?m)^Valor Total da Nota:[ \t]*R\$[ \t]*([\d.]+,\d{2})",
}

def brl_para_float(texto: str) -> float:
    return float(texto.replace(".", "").replace(",", "."))   # "12.345,67" → 12345.67

def extrair_nota(caminho: Path) -> NotaFiscal:
    try:
        texto = "\n".join(p.extract_text() or "" for p in PdfReader(caminho).pages)
    except Exception as e:                                    # arquivo corrompido
        raise ErroNegocio(f"PDF ilegível: {e}") from e
    # O extrator do PDF devolve "Rótulo: valor" colado; normalizamos espaços
    texto = re.sub(r"[ \t]+", " ", texto)
    dados = {}
    for campo, padrao in PADROES.items():
        m = re.search(padrao, texto)
        if not m:
            raise ErroNegocio(f"Campo '{campo}' não encontrado no documento")
        dados[campo] = m.group(1).strip()
    dados["valor"] = brl_para_float(dados["valor"])
    return NotaFiscal(arquivo=caminho.name, **dados)

# Teste com o primeiro PDF — e veja o texto "cru" que o robô enxerga
primeiro = sorted(PASTA_ENTRADA.glob("*.pdf"))[0]
print(PdfReader(primeiro).pages[0].extract_text()[:400], "\n...\n")
extrair_nota(primeiro)

## Parte 4 — Regras de negócio

As regras ficam **fora** da automação de tela: são funções puras, fáceis de testar. Um robô bem feito separa
**"o que decidir"** (regras) de **"como clicar"** (UI).

In [ ]:
# ============================ Validações de negócio ============================
STATUS_SUCESSO   = "SUCESSO"
STATUS_NEGOCIO   = "EXCECAO_NEGOCIO"
STATUS_SISTEMA   = "EXCECAO_SISTEMA"
STATUS_APROVACAO = "PENDENTE_APROVACAO"

def validar_nota(nota: NotaFiscal, chaves_ja_vistas: set) -> str | None:
    '''Retorna None se pode lançar, STATUS_APROVACAO se precisa de humano, ou levanta ErroNegocio.'''
    if not cnpj_valido(nota.cnpj):
        raise ErroNegocio(f"CNPJ inválido: {nota.cnpj}")
    chave = (nota.numero, limpar_cnpj(nota.cnpj))
    if chave in chaves_ja_vistas:
        raise ErroNegocio(f"NF {nota.numero} duplicada para o CNPJ {nota.cnpj}")
    chaves_ja_vistas.add(chave)
    if nota.valor <= 0:
        raise ErroNegocio("Valor da nota deve ser positivo")
    if nota.valor > LIMITE_APROVACAO:
        return STATUS_APROVACAO        # human-in-the-loop: robô prepara, pessoa decide
    return None

# Mini "teste unitário" das regras
_vistas = set()
_ok = NotaFiscal("x.pdf", "1", "01/10/2026", "A", gerar_cnpj(), "d", "TI", 100.0)
assert validar_nota(_ok, _vistas) is None
try:
    validar_nota(_ok, _vistas); raise AssertionError("deveria acusar duplicidade")
except ErroNegocio as e:
    print("✔ duplicidade detectada:", e)
print("✔ regras de negócio OK")

## Parte 5 — O robô no navegador (Playwright)

Agora as "mãos" do robô. Conceitos de automação de UI que aparecem aqui:

| Conceito | No código | Equivalente no UiPath |
|---|---|---|
| **Seletor** | `#txtCNPJ` | *Selector* / *Object Repository* |
| **Espera explícita** | `wait_for_selector(...)` | *Element Exists* / *Wait Element* |
| **Mapa de seletores** | dicionário `SELETORES` | *Object Repository* |
| **Evidência** | `page.screenshot(...)` | *Take Screenshot* |
| **Tratamento de tela de erro** | detectar `#erroSistema` | *Check App State* |

> ⚠️ **Por que a função `rodar(...)`?** O Jupyter já tem um *event loop* rodando e, **no Windows**, esse loop não consegue
> abrir o processo do navegador (o erro seria `NotImplementedError`). A função `rodar()` executa o robô numa *thread*
> com um loop próprio e compatível — funciona igual no Windows, Linux e Mac. Num script `.py` comum nada disso é preciso.

In [ ]:
# ============================ Camada de UI ============================
from playwright.async_api import async_playwright, TimeoutError as PWTimeout

def rodar(corrotina):
    '''Executa uma função async do robô num event loop próprio, em outra thread.
    Resolve o NotImplementedError do Playwright no Jupyter do Windows.'''
    saida = {}
    def alvo():
        loop = asyncio.ProactorEventLoop() if sys.platform == "win32" else asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        try:
            saida["valor"] = loop.run_until_complete(corrotina)
        except BaseException as e:
            saida["erro"] = e
        finally:
            loop.close()
    t = threading.Thread(target=alvo)
    t.start(); t.join()
    if "erro" in saida:
        raise saida["erro"]
    return saida.get("valor")

# Object Repository: TODO seletor mora aqui. Mudou a tela? Muda só este dicionário.
SELETORES = {
    "login_usuario": "#usuario",
    "login_senha":   "#senha",
    "login_botao":   "#btnEntrar",
    "home_ok":       "#boasVindas",
    "nf_numero":     "#txtNumNF",
    "nf_data":       "#txtDtEmissao",
    "nf_cnpj":       "#txtCNPJ",
    "nf_fornecedor": "#txtFornecedor",
    "nf_descricao":  "#txtDescricao",
    "nf_valor":      "#txtValor",
    "nf_cc":         "#cmbCentroCusto",
    "nf_gravar":     "#btnGravar",
    "res_protocolo": "#protocolo",
    "res_erro_val":  "#erroValidacao",
    "res_erro_sis":  "#erroSistema",
}

class RoboERP:
    '''Encapsula o navegador. Uso: async with RoboERP() as robo: await robo.lancar(nota)'''

    async def __aenter__(self):
        self._pw = await async_playwright().start()
        self.browser = await self._pw.chromium.launch(headless=HEADLESS, slow_mo=LENTIDAO_MS)
        self.page = await (await self.browser.new_context(locale="pt-BR")).new_page()
        self.page.set_default_timeout(8_000)          # nada de esperar "para sempre"
        await self.login()
        return self

    async def __aexit__(self, *exc):
        await self.browser.close()
        await self._pw.stop()

    async def login(self):
        p, s = self.page, SELETORES
        await p.goto(f"{ERP_URL}/login")
        await p.fill(s["login_usuario"], ERP_USUARIO)
        await p.fill(s["login_senha"], ERP_SENHA)
        await p.click(s["login_botao"])
        try:
            await p.wait_for_selector(s["home_ok"])
        except PWTimeout:
            # Senha errada não se resolve com retentativa → é de negócio (ou de configuração)
            raise ErroNegocio("Falha de login no ERP — verifique as credenciais do robô")
        log.info("🔐 Login realizado como %s", ERP_USUARIO)

    async def evidencia(self, nome: str) -> str:
        caminho = PASTA_EVID / f"{dt.datetime.now():%H%M%S}_{nome}.png"
        await self.page.screenshot(path=str(caminho), full_page=True)
        return str(caminho)

    async def lancar(self, nota: NotaFiscal) -> str:
        '''Preenche o formulário e devolve o protocolo do ERP.'''
        p, s = self.page, SELETORES
        await p.goto(f"{ERP_URL}/nova-nota")
        if "/login" in p.url:                          # sessão expirou? loga de novo
            await self.login(); await p.goto(f"{ERP_URL}/nova-nota")
        try:
            await p.wait_for_selector(s["nf_numero"])
        except PWTimeout:
            raise ErroSistema("Tela de lançamento não carregou (seletor não encontrado)")

        # TRANSFORMAÇÃO para o formato que o ERP aceita
        await p.fill(s["nf_numero"], nota.numero)
        await p.fill(s["nf_data"], nota.data_emissao)
        await p.fill(s["nf_cnpj"], limpar_cnpj(nota.cnpj))           # ERP quer só dígitos
        await p.fill(s["nf_fornecedor"], nota.prestador)
        await p.fill(s["nf_descricao"], nota.descricao)
        await p.fill(s["nf_valor"], f"{nota.valor:.2f}".replace(".", ","))   # 1234.5 → "1234,50"
        await p.select_option(s["nf_cc"], nota.centro_custo)
        await p.click(s["nf_gravar"])

        # Espera por QUALQUER um dos três desfechos possíveis
        desfecho = await p.wait_for_selector(
            f'{s["res_protocolo"]}, {s["res_erro_val"]}, {s["res_erro_sis"]}')
        id_el = await desfecho.get_attribute("id")
        texto = (await desfecho.inner_text()).strip()
        if id_el == s["res_protocolo"].lstrip("#"):
            return texto
        if id_el == s["res_erro_sis"].lstrip("#"):
            raise ErroSistema(texto)
        raise ErroNegocio(f"ERP recusou: {texto}")

print("Camada de UI pronta. Seletores mapeados:", len(SELETORES))

In [ ]:
# ============================ Teste isolado: 1 nota ============================
# Antes de rodar o lote, teste UMA transação. (Na vida real: em homologação, nunca em produção!)
async def teste_uma_nota():
    nota = extrair_nota(sorted(PASTA_ENTRADA.glob("*.pdf"))[0])
    async with RoboERP() as robo:
        for tentativa in range(1, 4):
            try:
                protocolo = await robo.lancar(nota)
                print(f"✅ Lançada na tentativa {tentativa}. Protocolo: {protocolo}")
                print("📸", await robo.evidencia("teste_unitario"))
                return
            except ErroSistema as e:
                print(f"⚠️ tentativa {tentativa}: {e}")

rodar(teste_uma_nota())

# Desfaz o teste para o lote começar do zero
ERP_DB.update(notas=[], seq=0)

## Parte 6 — Orquestração (padrão REFramework)

O "cérebro" do robô. Pontos de atenção que diferenciam um **robô de produção** de um **script que funciona na minha máquina**:

1. **Fila de transações** — cada PDF é um item independente; um erro não derruba o lote.
2. **Retentativa com *backoff* exponencial** — só para `ErroSistema` (0,5 s → 1 s → 2 s...).
3. **Idempotência** — se o robô cair no meio, rodar de novo **não pode lançar em duplicidade**.
4. ***Human-in-the-loop*** — notas acima do limite vão para uma fila de aprovação, não são lançadas.
5. **Evidência** de cada exceção (print de tela + log).

In [ ]:
# ============================ Orquestrador ============================
@dataclass
class Resultado:
    arquivo: str
    status: str
    numero: str = ""
    fornecedor: str = ""
    cnpj: str = ""
    valor: float | None = None
    protocolo: str = ""
    tentativas: int = 0
    mensagem: str = ""
    evidencia: str = ""
    duracao_s: float = 0.0

async def com_retentativas(funcao, *args, tentativas=MAX_TENTATIVAS, espera_base=0.5):
    '''Executa `funcao`; em ErroSistema/Timeout, espera e tenta de novo (backoff exponencial).'''
    for n in range(1, tentativas + 1):
        try:
            return await funcao(*args), n
        except (ErroSistema, PWTimeout) as e:
            if n == tentativas:
                raise ErroSistema(f"Falhou após {n} tentativas: {e}") from e
            espera = espera_base * 2 ** (n - 1)
            log.warning("   ↻ tentativa %d/%d falhou (%s). Nova tentativa em %.1fs", n, tentativas, e, espera)
            await asyncio.sleep(espera)

async def executar_robo() -> list[Resultado]:
    # ---------------- INIT ----------------
    inicio_lote = time.perf_counter()
    fila = sorted(PASTA_ENTRADA.glob("*.pdf"))          # Dispatcher: monta a fila
    resultados: list[Resultado] = []
    chaves_vistas: set = set()
    log.info("▶️  Início do lote: %d itens na fila", len(fila))

    async with RoboERP() as robo:
        # Idempotência: o que já está no ERP não pode ser lançado de novo
        ja_no_erp = {(n["numero"], n["cnpj"]) for n in ERP_DB["notas"]}  # num caso real: consultar a tela /notas

        # ---------------- GET TRANSACTION → PROCESS → SET STATUS ----------------
        for item in fila:
            t0 = time.perf_counter()
            r = Resultado(arquivo=item.name, status="")
            try:
                nota = extrair_nota(item)
                r.numero, r.fornecedor, r.cnpj, r.valor = nota.numero, nota.prestador, nota.cnpj, nota.valor

                if (nota.numero, limpar_cnpj(nota.cnpj)) in ja_no_erp:
                    raise ErroNegocio("Já existe no ERP (execução anterior) — ignorado por idempotência")

                if validar_nota(nota, chaves_vistas) == STATUS_APROVACAO:
                    r.status = STATUS_APROVACAO
                    r.mensagem = f"Valor R$ {brl(nota.valor)} acima do limite de R$ {brl(LIMITE_APROVACAO)}"
                else:
                    r.protocolo, r.tentativas = await com_retentativas(robo.lancar, nota)
                    r.status = STATUS_SUCESSO

            except ErroNegocio as e:
                r.status, r.mensagem = STATUS_NEGOCIO, str(e)
            except Exception as e:                        # ErroSistema esgotado ou falha inesperada
                r.status, r.mensagem = STATUS_SISTEMA, str(e)
                r.evidencia = await robo.evidencia(item.stem)

            r.duracao_s = round(time.perf_counter() - t0, 2)
            icone = {STATUS_SUCESSO: "✅", STATUS_NEGOCIO: "⛔", STATUS_SISTEMA: "💥", STATUS_APROVACAO: "🙋"}[r.status]
            log.info("%s %-22s %-19s %s", icone, item.name, r.status, r.protocolo or r.mensagem)
            resultados.append(r)

    # ---------------- END ----------------
    log.info("⏹️  Fim do lote em %.1fs", time.perf_counter() - inicio_lote)
    return resultados

resultados = rodar(executar_robo())
df = pd.DataFrame([asdict(r) for r in resultados])
df[["arquivo", "status", "numero", "valor", "protocolo", "tentativas", "mensagem"]]

> 🧪 **Experimento ao vivo:** rode a célula do robô **de novo**, sem gerar os PDFs. O que acontece com as notas que já estavam no ERP?
> Isso é **idempotência**. Depois mude `HEADLESS = False` e `LENTIDAO_MS = 400` na configuração, rode tudo de novo e veja o robô "digitando".

## Parte 7 — Relatório para o negócio e cálculo de ROI

O gestor não quer ver log: quer uma **planilha** com o que foi feito, o que precisa de atenção e quanto se economizou.

In [ ]:
# ============================ Relatório Excel ============================
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

# Premissas de ROI — DISCUTA com a turma se são realistas!
MIN_HUMANO_POR_NOTA = 4.0        # minutos que o analista gasta por nota
NOTAS_POR_MES       = 800
CUSTO_HORA_ANALISTA = 45.00      # R$/h com encargos (hipotético)
CUSTO_MENSAL_ROBO   = 1_500.00   # R$/mês: infraestrutura + manutenção (hipotético)

total = len(df)
ok = (df.status == STATUS_SUCESSO).sum()
taxa_auto = ok / total
seg_robo = df.loc[df.status == STATUS_SUCESSO, "duracao_s"].mean()
horas_economizadas = NOTAS_POR_MES * taxa_auto * MIN_HUMANO_POR_NOTA / 60
economia_mes = horas_economizadas * CUSTO_HORA_ANALISTA - CUSTO_MENSAL_ROBO

resumo = pd.DataFrame({
    "Indicador": ["Notas processadas", "Lançadas pelo robô", "Taxa de automação (STP)",
                  "Exceções de negócio", "Exceções de sistema", "Aguardando aprovação",
                  "Tempo médio do robô por nota (s)", "Tempo humano por nota (s)",
                  "Horas liberadas por mês (projeção)", "Economia líquida mensal (R$, projeção)"],
    "Valor": [total, ok, f"{taxa_auto:.0%}",
              (df.status == STATUS_NEGOCIO).sum(), (df.status == STATUS_SISTEMA).sum(),
              (df.status == STATUS_APROVACAO).sum(),
              round(seg_robo, 1), MIN_HUMANO_POR_NOTA * 60,
              round(horas_economizadas, 1), f"R$ {brl(economia_mes)}"],
})

arquivo_rel = PASTA_SAIDA / f"relatorio_robo_AP_{dt.datetime.now():%Y%m%d_%H%M}.xlsx"
with pd.ExcelWriter(arquivo_rel, engine="openpyxl") as xw:
    resumo.to_excel(xw, sheet_name="Resumo", index=False)
    df[df.status == STATUS_SUCESSO].to_excel(xw, sheet_name="Lançadas", index=False)
    df[df.status != STATUS_SUCESSO].to_excel(xw, sheet_name="Para analisar", index=False)
    for ws in xw.book.worksheets:                         # formatação: cabeçalho + largura
        for cel in ws[1]:
            cel.font = Font(bold=True, color="FFFFFF")
            cel.fill = PatternFill("solid", fgColor="1E4D45")
            cel.alignment = Alignment(horizontal="center")
        for col in ws.columns:
            largura = max(len(str(c.value or "")) for c in col) + 2
            ws.column_dimensions[get_column_letter(col[0].column)].width = min(largura, 60)
        ws.freeze_panes = "A2"

print("📊 Relatório salvo em", arquivo_rel.resolve())
resumo

> 🗣️ **Discussão:** neste lote "plantamos" 4 problemas em 12 notas, então a taxa de automação ficou em ~67% e a economia líquida
> quase zerou. **Moral:** o ROI de RPA depende muito mais da **qualidade do processo e dos dados de entrada** do que da velocidade do robô.
> Altere as premissas (volume, custo/hora, % de exceções) e encontre o *ponto de equilíbrio*. A partir de que taxa de STP o projeto se paga?

## Parte 8 — Auditoria: o robô fez o que disse que fez?

Todo robô de produção precisa de **reconciliação**: comparar o que o robô *registrou* com o que de fato está no *sistema-alvo*.
Auditores (e a LGPD/SOX) vão perguntar isso.

In [ ]:
# ============================ Reconciliação robô × ERP ============================
no_erp = pd.DataFrame(ERP_DB["notas"])
lancadas_robo = df[df.status == STATUS_SUCESSO]

prot_robo = set(lancadas_robo.protocolo)
prot_erp = set(no_erp.protocolo) if not no_erp.empty else set()

print(f"Robô diz que lançou: {len(prot_robo)} | ERP tem: {len(prot_erp)}")
print("Divergências:", (prot_robo ^ prot_erp) or "nenhuma ✅")

# Bate também os VALORES (erro de transformação é o bug mais comum de RPA!)
if not no_erp.empty:
    conf = lancadas_robo.merge(no_erp[["protocolo", "valor"]], on="protocolo", suffixes=("_pdf", "_erp"))
    conf["valor_erp_num"] = conf["valor_erp"].str.replace(",", ".").astype(float)
    conf["confere"] = (conf["valor_pdf"] - conf["valor_erp_num"]).abs() < 0.01
    display(conf[["numero", "protocolo", "valor_pdf", "valor_erp", "confere"]])

---
## Parte 9 — 🏆 Atividade avaliativa (em dupla)

Agora é com vocês. Cada desafio abaixo tem **enunciado + célula pronta para completar** (procure por `# TODO`).
Ao terminar cada um, escreva um **parágrafo de reflexão** na célula de texto logo abaixo dele.

**Nota:** desafios 1 a 4 valem 2,5 pontos cada (máximo 10); o desafio 5 vale até 1,0 ponto extra.

| Critério (vale para cada desafio) | Peso |
|---|---|
| Funciona (roda do início ao fim sem erro; os testes passam) | 40% |
| Tratamento correto de exceções (negócio × sistema) | 25% |
| Código legível e comentado | 15% |
| Reflexão (argumenta, não só descreve o código) | 20% |

**Entrega:** este notebook `.ipynb`, com os nomes da dupla na célula abaixo.

In [ ]:
DUPLA = ["Nome 1", "Nome 2"]   # TODO: preencha

### Desafio 1 — CNPJ alfanumérico (notícia real!) · 2,5 pts
Desde **31/07/2026** a Receita Federal emite **CNPJ alfanumérico** (IN RFB 2.229/2024): as 12 primeiras posições podem ter
**letras e números**; os 2 dígitos verificadores continuam numéricos. O DV usa o valor `ASCII − 48` de cada caractere (`A` = 17).

1. Implemente `limpar_cnpj_v2` e `cnpj_valido_v2` aceitando o novo formato (sem quebrar os numéricos).
2. Ajuste o padrão `PADROES["cnpj"]` da extração para aceitar letras.
3. Faça os três testes passarem (tire o `#` deles).

In [ ]:
# ============================ DESAFIO 1 ============================
# Dica: _calcular_dv já usa ord(c) - 48. O problema está em quem "limpa" o CNPJ e no regex.

def limpar_cnpj_v2(cnpj: str) -> str:
    ...  # TODO: manter dígitos E letras (em maiúsculas), removendo . / -

def cnpj_valido_v2(cnpj: str) -> bool:
    ...  # TODO: mesmo algoritmo de cnpj_valido, usando limpar_cnpj_v2

# TODO: ajuste o regex para aceitar letras
# PADROES["cnpj"] = r"..."

# Testes (tire o # quando terminar):
# assert cnpj_valido_v2("12.ABC.345/01DE-35")        # exemplo oficial da Receita
# assert not cnpj_valido_v2("12.ABC.345/01DE-36")
# assert cnpj_valido_v2("23.775.684/0001-07")        # numéricos continuam válidos

**Reflexão 1:** quantos robôs no Brasil têm `[0-9]` "chumbado" no código? Que tipo de exceção isso gera em produção, e como você perceberia?

### Desafio 2 — E-mail de fim de lote · 2,5 pts
Gere um **e-mail** para a gerência (arquivo `.eml`, que abre no Outlook/Thunderbird) com:
- assunto com a data e o total de notas;
- no corpo: o resumo (lançadas, exceções, aprovações) e **a lista das notas que precisam de ação humana**;
- o relatório Excel (`arquivo_rel`) **em anexo**.

In [ ]:
# ============================ DESAFIO 2 ============================
from email.message import EmailMessage

def gerar_email(df, arquivo_excel) -> Path:
    msg = EmailMessage()
    msg["From"] = "robo.ap@transvia.com.br"
    msg["To"] = "gerencia.ap@transvia.com.br"
    msg["Subject"] = "..."          # TODO: data + total de notas
    corpo = "..."                    # TODO: resumo + lista de notas com status != SUCESSO
    msg.set_content(corpo)
    # TODO: anexar o Excel com msg.add_attachment(dados_em_bytes, maintype="application",
    #        subtype="vnd.openxmlformats-officedocument.spreadsheetml.sheet", filename=arquivo_excel.name)
    caminho = PASTA_SAIDA / "resumo_lote.eml"
    caminho.write_bytes(bytes(msg))
    return caminho

# print("📧 E-mail gerado em", gerar_email(df, arquivo_rel))

**Reflexão 2:** por que o robô deve avisar *também* quando tudo deu certo? O que um gestor precisa ver em 10 segundos?

### Desafio 3 — Robô à prova de queda · 2,5 pts
Hoje a idempotência lê `ERP_DB` diretamente — um atalho de laboratório (num ERP real você não tem acesso ao banco!).
Torne-a **real**: o robô deve abrir a tela `/notas`, ler a tabela `#tblNotas` e devolver o conjunto `{(numero, cnpj)}` já lançado.
Depois, use essa função no lugar de `ja_no_erp = {...}` dentro de `executar_robo()` e prove que rodar o lote duas vezes não duplica nada.

In [ ]:
# ============================ DESAFIO 3 ============================
async def ler_notas_do_erp(robo) -> set:
    p = robo.page
    await p.goto(f"{ERP_URL}/notas")
    linhas = await p.locator("#tblNotas tr").all()
    lancadas = set()
    for linha in linhas[1:]:                         # pula o cabeçalho
        celulas = await linha.locator("td").all_inner_texts()
        ...  # TODO: colunas = [Protocolo, NF, CNPJ, ...] → adicionar (NF, CNPJ) em lancadas
    return lancadas

async def testar_desafio3():
    async with RoboERP() as robo:
        return await ler_notas_do_erp(robo)

# lidas = rodar(testar_desafio3())
# assert lidas == {(n["numero"], n["cnpj"]) for n in ERP_DB["notas"]}, "a tela e o ERP deveriam bater"
# print("✔", len(lidas), "notas lidas pela tela")

**Reflexão 3:** por que ler pela tela é mais lento, porém mais correto, do que confiar numa lista guardada pelo próprio robô?

### Desafio 4 — "Atualizaram o sistema e o robô parou" · 2,5 pts
1. Mude `LAYOUT_ERP = "v2"` na célula de configuração e rode de novo a configuração e a Parte 2 (o ERP). Rode o robô e veja as falhas.
2. **Parte A:** abra `http://127.0.0.1:8765/nova-nota`, inspecione os novos IDs (F12) e crie `SELETORES_V2` — mudando **só o dicionário**.
3. **Parte B:** torne o robô resiliente usando **rótulos** em vez de IDs (`page.get_by_label("CNPJ fornecedor")`),
   de modo que o mesmo código funcione no v1 **e** no v2. Complete `lancar_por_rotulo` abaixo.

In [ ]:
# ============================ DESAFIO 4 ============================
# Parte A — só o dicionário muda
SELETORES_V2 = dict(SELETORES)
# TODO: SELETORES_V2["nf_numero"] = "#..."   (e os demais campos e o botão)

# Parte B — seletores por rótulo, independentes de ID
async def lancar_por_rotulo(robo, nota) -> str:
    p = robo.page
    await p.goto(f"{ERP_URL}/nova-nota")
    await p.get_by_label("Nº da NF").fill(nota.numero)
    await p.get_by_label("CNPJ fornecedor").fill(limpar_cnpj(nota.cnpj))
    ...  # TODO: os demais campos (Emissão, Razão social, Descrição, Valor, Centro de custo)
    await p.get_by_role("button").click()            # funciona com "Gravar" e "Salvar lançamento"
    ...  # TODO: esperar o protocolo ou o erro, como em RoboERP.lancar, e devolver o protocolo

**Reflexão 4:** por que centralizar os seletores num só lugar? Quais as vantagens e riscos de usar rótulos em vez de IDs?

### Desafio 5 (extra) — Um passo "agêntico" · até +1,0 ponto
Algumas notas chegam **sem centro de custo**. Crie `classificar_centro_custo(descricao)` que devolve `(centro, confianca)`
a partir da descrição — primeiro com palavras-chave; depois (opcional) chamando um LLM por API.
**Regra obrigatória:** com confiança baixa, a nota vai para a **fila de aprovação humana**, nunca para um "chute".

In [ ]:
# ============================ DESAFIO 5 (extra) ============================
PALAVRAS_CHAVE = {
    "OPERACOES":      ["frota", "frete", "caminh"],
    "TI":             ["licença", "sistema", "software", "gps"],
    "ENGENHARIA":     ["sinalização", "laudo", "recapeamento", "estrutural"],
    "ADMINISTRATIVO": ["limpeza", "ar-condicionado", "conservação"],
}

def classificar_centro_custo(descricao: str) -> tuple[str | None, float]:
    ...  # TODO: contar palavras-chave por centro; confiança = acertos do vencedor / total de acertos
    return None, 0.0

LIMIAR_CONFIANCA = 0.7
# for d in ["Manutenção preventiva de frota", "Licença de software de GPS", "Consultoria diversa"]:
#     cc, conf = classificar_centro_custo(d)
#     destino = cc if conf >= LIMIAR_CONFIANCA else "→ fila de aprovação humana"
#     print(f"{d:<40} {destino}  (confiança {conf:.0%})")

**Reflexão 5:** o que muda em termos de **auditoria** quando a decisão deixa de ser uma regra fixa e passa a ser probabilística?

In [ ]:
# ============================ Encerramento ============================
# Desliga o ERP de laboratório (rode ao final da aula)
# SERVIDOR_ERP.shutdown(); SERVIDOR_ERP.server_close(); print("ERP desligado")